# Advanced Statistics — Maintenance Assignment
Google Colab notebook; upload the CSV when prompted.
Sample: 300 random rows; population: 2400 records; cluster-aware checks for repeated sections.

In [1]:
# Advanced Statistics - Maintenance Assignment
# Google Colab | style similar to professor's scipy.stats examples
import numpy as np
import pandas as pd
from scipy import stats
from google.colab import files

print('Upload pavement_transportation_master_dataset.csv')
uploaded = files.upload()
filename = next((name for name in uploaded if name.endswith('.csv')), None)
if filename is None:
    raise ValueError('Please upload the CSV dataset.')
df = pd.read_csv(filename)
print('Dataset:', df.shape, '| Road sections:', df.section_id.nunique())

# Random 300 observations from the full dataset (not restricted to 2025)
np.random.seed(42)
sample = df.sample(n=300, random_state=42).copy()
# For independent 300-section analysis, choose one observation per section instead:
# sample = df.sort_values('survey_year').groupby('section_id', group_keys=False).sample(n=1, random_state=42)

alpha = 0.05

def decision(p):
    return 'Reject H0' if p < alpha else 'Fail to Reject H0'

Upload pavement_transportation_master_dataset.csv


Saving pavement_transportation_master_dataset.csv to pavement_transportation_master_dataset (1).csv
Dataset: (2400, 33) | Road sections: 300


In [5]:
# Q1: Is mean maintenance cost different from $4000?
def question_1(data, title):
    x = data['maintenance_cost_USD'].dropna()
    print('\n1. One-Sample t-Test:', title)
    print(f'n={len(x)}, mean={x.mean():.2f}, median={x.median():.2f}, SD={x.std():.2f}')
    w, p_normal = stats.shapiro(x)
    print(f'Shapiro-Wilk: W={w:.4f}, p={p_normal:.6g}')
    t, p = stats.ttest_1samp(x, popmean=4000)
    print(f't={t:.4f}, p-value={p:.6g}, Decision: {decision(p)}')
    ci = stats.t.interval(0.95, len(x)-1, loc=x.mean(), scale=stats.sem(x))
    print(f'95% CI for mean: ({ci[0]:.2f}, {ci[1]:.2f})')

# Q2: Do High and Low priority differ in maintenance COST?
def question_2(data, title):
    high = data.loc[data.maintenance_priority == 'High', 'maintenance_cost_USD'].dropna()
    low = data.loc[data.maintenance_priority == 'Low', 'maintenance_cost_USD'].dropna()
    print('\n2. Mann-Whitney U Test:', title)
    print(f'High: n={len(high)}, mean={high.mean():.2f}, median={high.median():.2f}')
    print(f'Low:  n={len(low)}, mean={low.mean():.2f}, median={low.median():.2f}')
    u, p = stats.mannwhitneyu(high, low, alternative='two-sided')
    print(f'U={u:.4f}, p-value={p:.6g}, Decision: {decision(p)}')

# Q3: Does maintenance type differ in maintenance COST?
def question_3(data, title):
    typed = data.dropna(subset=['maintenance_type', 'maintenance_cost_USD'])
    groups = [g.maintenance_cost_USD.to_numpy() for _, g in typed.groupby('maintenance_type')]
    print('\n3. One-Way ANOVA and Kruskal-Wallis:', title)
    print('Valid maintenance types:', len(typed), 'out of', len(data))
    print(typed.groupby('maintenance_type').maintenance_cost_USD.agg(['count', 'mean', 'median']).to_string())
    if len(groups) < 2 or min(map(len, groups)) < 2:
        print('Insufficient observations for group comparison.')
        return
    lev, p_lev = stats.levene(*groups, center='median')
    print(f'Levene: stat={lev:.4f}, p={p_lev:.6g}')
    f, p = stats.f_oneway(*groups)
    print(f'ANOVA: F={f:.4f}, p-value={p:.6g}, Decision: {decision(p)}')
    h, p_kw = stats.kruskal(*groups)
    print(f'Kruskal-Wallis: H={h:.4f}, p-value={p_kw:.6g}, Decision: {decision(p_kw)}')

In [3]:
print('\n======== RANDOM SAMPLE: 300 ROWS ========')
question_1(sample, 'Random 300 rows')
question_2(sample, 'Random 300 rows')
question_3(sample, 'Random 300 rows')

print('\n======== ALL 2400 RECORDS: DESCRIPTIVE / NAIVE TESTS ========')
question_1(df, 'All 2400 rows (naive independence)')
question_2(df, 'All 2400 rows (naive independence)')
question_3(df, 'All 2400 rows (naive independence)')


======== RANDOM SAMPLE: 300 ROWS ========

1. One-Sample t-Test: Random 300 rows
n=300, mean=4087.86, median=730.26, SD=9933.34
Shapiro-Wilk: W=0.4410, p=1.62807e-29
t=0.1532, p-value=0.878346, Decision: Fail to Reject H0
95% CI for mean: (2959.25, 5216.47)

2. Mann-Whitney U Test: Random 300 rows
High: n=116, mean=1999.46, median=699.42
Low:  n=32, mean=16415.08, median=2881.43
U=1323.5000, p-value=0.0110703, Decision: Reject H0

3. One-Way ANOVA and Kruskal-Wallis: Random 300 rows
Valid maintenance types: 29 out of 300
                  count          mean    median
maintenance_type                               
Preventive           14  17557.461429  17201.09
Reconstruction        7  50201.584286  50534.56
Rehabilitation        8  37101.112500  36084.62
Levene: stat=0.1260, p=0.882135
ANOVA: F=143.3773, p-value=9.05747e-15, Decision: Reject H0
Kruskal-Wallis: H=23.8966, p-value=6.47038e-06, Decision: Reject H0

======== ALL 2400 RECORDS: DESCRIPTIVE / NAIVE TESTS ========

1. One-S

In [4]:
# Repeated observations of the same section are dependent.
# Q1: test the 300 independent section-level means.
section_means = df.groupby('section_id', as_index=False)['maintenance_cost_USD'].mean()
print('\n======== CLUSTER-AWARE FULL-DATA ANALYSIS ========')
question_1(section_means, '300 independent section means')

# Q2 and Q3: cluster-robust standard errors by section_id.
# This section uses statsmodels only where scipy.stats is insufficient.
import statsmodels.formula.api as smf
q2 = df[df.maintenance_priority.isin(['High', 'Low'])].dropna(subset=['maintenance_cost_USD'])
fit2 = smf.ols('maintenance_cost_USD ~ C(maintenance_priority, Treatment(reference="Low"))', data=q2).fit(
    cov_type='cluster', cov_kwds={'groups': q2.section_id})
print('\nQ2: Full data, cluster-robust High vs Low difference')
print(fit2.summary().tables[1])

q3 = df.dropna(subset=['maintenance_type', 'maintenance_cost_USD']).copy()
fit3 = smf.ols('maintenance_cost_USD ~ C(maintenance_type)', data=q3).fit(
    cov_type='cluster', cov_kwds={'groups': q3.section_id})
print('\nQ3: Full data, cluster-robust maintenance-type model')
print(fit3.summary().tables[1])
terms = [x for x in fit3.params.index if x != 'Intercept']
restriction = ', '.join(f'{x} = 0' for x in terms)
print('Joint Wald test:', fit3.wald_test(restriction, scalar=True))
print('\nNote: Associations are not causal effects; maintenance_type has many missing values.')


======== CLUSTER-AWARE FULL-DATA ANALYSIS ========

1. One-Sample t-Test: 300 independent section means
n=300, mean=3722.62, median=2500.62, SD=3233.94
Shapiro-Wilk: W=0.8565, p=4.93383e-16
t=-1.4856, p-value=0.138443, Decision: Fail to Reject H0
95% CI for mean: (3355.19, 4090.06)

Q2: Full data, cluster-robust High vs Low difference
                                                                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------------------------
Intercept                                                    1.114e+04   1141.375      9.757      0.000    8899.273    1.34e+04
C(maintenance_priority, Treatment(reference="Low"))[T.High] -8473.7945   1170.693     -7.238      0.000   -1.08e+04   -6179.279

Q3: Full data, cluster-robust maintenance-type model
                                            coef    std err          z      P>|z|      [0.025   